In [6]:
import pandas as pd
import numpy as np
import scipy
import os
import kagglehub
import sklearn

In [16]:
data = sklearn.datasets.load_breast_cancer()
df = pd.DataFrame(data['data'], columns=data['feature_names'])
df['target'] = data['target']

In [17]:
print(df.columns)

Index(['mean radius', 'mean texture', 'mean perimeter', 'mean area',
       'mean smoothness', 'mean compactness', 'mean concavity',
       'mean concave points', 'mean symmetry', 'mean fractal dimension',
       'radius error', 'texture error', 'perimeter error', 'area error',
       'smoothness error', 'compactness error', 'concavity error',
       'concave points error', 'symmetry error', 'fractal dimension error',
       'worst radius', 'worst texture', 'worst perimeter', 'worst area',
       'worst smoothness', 'worst compactness', 'worst concavity',
       'worst concave points', 'worst symmetry', 'worst fractal dimension',
       'target'],
      dtype='str')


In [27]:
def test_leakage_binary(df: pd.DataFrame, target: str):
    leak_numerical = {}
    leak_categorical = {}
    for col_other in list(df.columns):
        mask = df[col_other].notna() & df[target].notna()
        if mask.sum() < 2:
            continue
        if target == col_other:
            continue
        elif df[col_other].dtype in (int, float):
            df_masked = df.loc[mask]
            r, p_val = scipy.stats.pointbiserialr(df_masked[target], df_masked[col_other])
            if (abs(r) > 0.9) & (p_val < 0.05):
                leak_numerical[(target, col_other)] = p_val
        else:
            df_masked = df.loc[mask]
            crosstab = pd.crosstab(df_masked[col_other], df_masked[target])
            p_val = scipy.stats.chi2_contingency(crosstab)[1]
            if p_val < 0.05:
                leak_categorical[(target, col_other)] = p_val

    return leak_numerical, leak_categorical

leak_n, leak_c = test_leakage_binary(df, 'target')

print(leak_n)
print(leak_c)

{}
{}


In [29]:
df = pd.read_csv('hour.csv')
print(df.columns)

Index(['instant', 'dteday', 'season', 'yr', 'mnth', 'hr', 'holiday', 'weekday',
       'workingday', 'weathersit', 'temp', 'atemp', 'hum', 'windspeed',
       'casual', 'registered', 'cnt'],
      dtype='str')


In [36]:
def test_leakage_continuous(df:pd.DataFrame, target:str) -> dict:
    leakage = {}
    for col in list(df.columns):
        mask = (df[target].notnull()) & (df[col].notnull())
        if mask.sum() < 3:
            continue
        if col == target:
            continue
        elif df[col].dtype in (int, float):
            r = scipy.stats.pearsonr(df.loc[mask][target], df.loc[mask][col]).statistic
            if abs(r) > 0.9:
                leakage[(target, col)] = r

    return leakage

leakage = test_leakage_continuous(df, 'cnt')
print(leakage)


{('cnt', 'registered'): np.float64(0.9721507308642993)}
